# Alpha-Segment Latent AOPC Sweep

Notebook для сравнения `IG`, `NAA` и одного фиксированного `Cheap-IG` по зависимости `latent deletion AOC@20%` от правой границы alpha-отрезка `segment_end`.

Эксперимент classifier-only, работает на `yolo11s-cls`, использует raw neurons `(c,h,w)`, считает только `deletion`, а attribution baseline фиксирует как `zero`.

## Импорты

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from modules.alpha_segment_benchmark import (
    benchmark_classifier_alpha_segment_latent_aopc,
    default_classifier_method_specs,
    render_alpha_segment_report,
)


## Параметры

In [2]:
OXFORD_PETS_DIR = Path("oxford_pets")
N_IMAGES = 100

CLASSIFIER_LAYER = "model.6"
N_STEPS = 192
SEGMENT_END_VALUES = [round(step / 10.0, 1) for step in range(1, 11)]
BUDGET_PERCENTILES = list(range(1, 21))
DONOR_KINDS = [
    "zero_baseline",
    "black_act",
    "blur_act",
    "layer_mean_exclusive",
    "spatial_nli_same_channel",
]
BLUR_SIGMA = 16.0

CHEAP_IG_CONFIG = {
    "selection_mode": "positive",
    "selection_top_k": 4000,
    "fill_mode": "naa_scaled",
    "fill_rho": 0.6,
}

VISUAL_IMAGES = 10
CLEAR_EVERY = 8
FD_EPS = 1e-3

CACHE_ROOT = Path("output/alpha_segment_cache")
OUTPUT_DIR = Path("output/alpha_segment_latent_aopc_oxford_pets_100")

REFRESH_CORE = False
REFRESH_METHODS = False
REFRESH_EVALUATIONS = False


In [3]:
def collect_oxford_pets_images(image_dir=OXFORD_PETS_DIR, n_images=N_IMAGES):
    image_dir = Path(image_dir)
    if not image_dir.exists():
        raise FileNotFoundError(f"Directory not found: {image_dir}")

    image_paths = []
    for pattern in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        image_paths.extend(image_dir.glob(pattern))

    image_paths = sorted(set(image_paths), key=lambda path: path.name.lower())
    if len(image_paths) < n_images:
        raise RuntimeError(f"Expected at least {n_images} images in {image_dir}, found {len(image_paths)}")
    return [str(path) for path in image_paths[:n_images]]


IMAGE_PATHS = collect_oxford_pets_images()
len(IMAGE_PATHS), IMAGE_PATHS[:5]


(100,
 ['oxford_pets/Abyssinian_1.jpg',
  'oxford_pets/Abyssinian_108.jpg',
  'oxford_pets/Abyssinian_117.jpg',
  'oxford_pets/Abyssinian_126.jpg',
  'oxford_pets/Abyssinian_135.jpg'])

## Методы

In [4]:
METHOD_SPECS = default_classifier_method_specs(
    segment_end_values=SEGMENT_END_VALUES,
    cheap_ig_config=CHEAP_IG_CONFIG,
)

len(METHOD_SPECS), [spec["name"] for spec in METHOD_SPECS[:6]]


(30,
 ['IG[0,0.1]',
  'NAA[0,0.1]',
  'cheap-ig[0,0.1]/positive/k4000/fill-naa_scaled-rho0.6',
  'IG[0,0.2]',
  'NAA[0,0.2]',
  'cheap-ig[0,0.2]/positive/k4000/fill-naa_scaled-rho0.6'])

## Запуск Benchmark

In [5]:
results = benchmark_classifier_alpha_segment_latent_aopc(
    image_paths=IMAGE_PATHS,
    method_specs=METHOD_SPECS,
    layer_name=CLASSIFIER_LAYER,
    n_steps=N_STEPS,
    budget_percentiles=BUDGET_PERCENTILES,
    donor_kinds=DONOR_KINDS,
    blur_sigma=BLUR_SIGMA,
    visual_images=VISUAL_IMAGES,
    cache_root=CACHE_ROOT,
    output_dir=OUTPUT_DIR,
    fd_eps=FD_EPS,
    clear_every=CLEAR_EVERY,
    refresh_core=REFRESH_CORE,
    refresh_methods=REFRESH_METHODS,
    refresh_evaluations=REFRESH_EVALUATIONS,
    verbose=False,
)

artifacts = render_alpha_segment_report(results, output_dir=OUTPUT_DIR)
artifacts["report_path"], artifacts["summary_path"]


('output/alpha_segment_latent_aopc_oxford_pets_100/alpha_segment_report.md',
 'output/alpha_segment_latent_aopc_oxford_pets_100/alpha_segment_summary.json')

## Отчёт

In [ ]:
display(Markdown(Path(artifacts["report_path"]).read_text(encoding="utf-8")))


# Alpha-Segment Latent AOPC Sweep

Classifier-only benchmark for `yolo11s-cls` with raw-neuron deletion AOC20 in latent space.

## Configuration

- layer_name=`model.6`
- n_steps=`192`
- budget_percentiles=`[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]`
- segment_end_values=`[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]`
- donor_kinds=`['zero_baseline', 'black_act', 'blur_act', 'layer_mean_exclusive', 'spatial_nli_same_channel']`
- blur_sigma=`16.0`
- n_images=`100`

## Peak Summary

| Method | Donor | score@0.1 | best_end | peak_contrast | AOC20 mean(all ends) | AOC20 norm mean(all ends) |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Cheap-IG | black_act | 14.6163 | 1.00 | -0.7863 | 15.3240 | 1.0802 |
| Cheap-IG | blur_act | 13.3453 | 0.90 | -1.0341 | 14.2760 | 1.0053 |
| Cheap-IG | layer_mean_exclusive | 13.5492 | 1.00 | -1.0011 | 14.4502 | 1.0183 |
| Cheap-IG | spatial_nli_same_channel | 12.4772 | 0.90 | -1.4261 | 13.7607 | 0.9713 |
| Cheap-IG | zero_baseline | 13.2555 | 0.90 | -0.9879 | 14.1446 | 0.9967 |
| IG | black_act | 11.1153 | 1.00 | -2.8324 | 13.6644 | 0.9590 |
| IG | blur_act | 10.8771 | 1.00 | -2.6335 | 13.2472 | 0.9305 |
| IG | layer_mean_exclusive | 10.1474 | 1.00 | -2.9725 | 12.8226 | 0.9004 |
| IG | spatial_nli_same_channel | 8.8951 | 0.90 | -3.8347 | 12.3463 | 0.8686 |
| IG | zero_baseline | 9.6959 | 1.00 | -3.0690 | 12.4580 | 0.8744 |
| NAA | black_act | 12.8684 | 1.00 | -2.8391 | 15.4235 | 1.0849 |
| NAA | blur_act | 11.1279 | 1.00 | -3.4803 | 14.2602 | 1.0025 |
| NAA | layer_mean_exclusive | 11.5092 | 1.00 | -3.4089 | 14.5771 | 1.0251 |
| NAA | spatial_nli_same_channel | 10.5246 | 1.00 | -4.0912 | 14.2067 | 0.9999 |
| NAA | zero_baseline | 11.1433 | 1.00 | -3.4885 | 14.2830 | 1.0043 |

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/peak_summary.png)

## IG

- best donor by aggregate metric: `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/IG_aggregate_curves.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/IG_aggregate_curves_normalized.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/IG_single_image_curves.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/IG_donor_vs_segment_heatmap.png)

### Visual Tables

Page 1 | best donor `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/IG_visual_page_1.png)

Page 2 | best donor `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/IG_visual_page_2.png)

## NAA

- best donor by aggregate metric: `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/NAA_aggregate_curves.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/NAA_aggregate_curves_normalized.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/NAA_single_image_curves.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/NAA_donor_vs_segment_heatmap.png)

### Visual Tables

Page 1 | best donor `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/NAA_visual_page_1.png)

Page 2 | best donor `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/NAA_visual_page_2.png)

## Cheap-IG

- best donor by aggregate metric: `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/Cheap_IG_aggregate_curves.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/Cheap_IG_aggregate_curves_normalized.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/Cheap_IG_single_image_curves.png)

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/Cheap_IG_donor_vs_segment_heatmap.png)

### Visual Tables

Page 1 | best donor `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/Cheap_IG_visual_page_1.png)

Page 2 | best donor `black_act`

![](output/alpha_segment_latent_aopc_oxford_pets_100/figures/Cheap_IG_visual_page_2.png)



: 